# GBD Vietnam - Analysis Runner

Clean top-to-bottom execution of the full pipeline.
Each cell executes one numbered script from `scripts/`.

The individual scripts are also runnable standalone from the
terminal, e.g. `python scripts/01_load_clean.py`.


## 1. Setup

Resolve paths, ensure output folders, verify raw inputs.

In [1]:
import importlib.util, sys
from pathlib import Path

SCRIPTS = Path('scripts').resolve()
sys.path.insert(0, str(SCRIPTS))

def load_script(numbered_name):
    """Import a numbered script (e.g. '01_load_clean') as a module."""
    path = SCRIPTS / f'{numbered_name}.py'
    spec = importlib.util.spec_from_file_location(numbered_name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

setup = load_script('00_setup')
setup.run()



=== 00 SETUP ===
  project root: D:\gbd
  output dirs ok: processed, figures/html, figures/static, tables
  all 9 expected raw files present


[]

## 2. Load & Clean

Read raw CSVs (including `query5_yll_yld.csv`), filter to SEA + 1990-2023, save to `data/processed/`.

In [2]:
load_clean = load_script('01_load_clean')
df_burden, df_yll_yld, df_pop = load_clean.run()



=== 01 LOAD & CLEAN ===


  Burden rows:  9,248  (11 countries)
  YLL/YLD rows: 6,732  (11 countries)
  Years: 1990-2023
  Missing in burden val/lower/upper: 0
  [ok] cleaned files written to data/processed/


## 3. Core Metrics

Cause composition (CMNN/NCD/Injuries shares), 30q70, YLL/YLD ratio, and Table 1 summary.

In [3]:
metrics = load_script('02_metrics')
m = metrics.run()
yll_yld_ratio = m['yll_yld']



=== 02 METRICS ===


  Vietnam NCD share 1990->2023: 52.99% -> 70.67%
  Vietnam 30q70 NCD 1990->2023: 25.02% -> 21.80%
  YLL/YLD ratio 1990: 2.85
  YLL/YLD ratio 2023: 1.84
  [ok] tables/metrics.csv, tables/table1_summary.csv, data/processed/probability_30q70.csv
  -- CMNN sensitivity analysis --


  CMNN split max reconciliation error: 0.000
  NCD share (main)         2023 = 70.67%
  NCD share vs C-only      2023 = 74.30%
  NCD share vs M+N+N only  2023 = 78.39%
  [ok] tables/table_s1_cmnn_sensitivity.csv, data/processed/cmnn_split.csv


## 4. Trends

AAPC (log-linear regression) for CMNN/NCD/Injuries DALY rate and YLL/YLD rates; full-period + three decades.

In [4]:
trends = load_script('03_trends')
df_trends = trends.run()



=== 03 TRENDS ===


  Vietnam AAPC 1990-2023:
    DALY - CMNN                  AAPC = -4.63% (95% CI -4.80, -4.46; p=0)
    DALY - NCD                   AAPC = -0.38% (95% CI -0.45, -0.30; p=0)
    DALY - Injuries              AAPC = -1.13% (95% CI -1.24, -1.03; p=0)
    YLL rate - All causes        AAPC = -1.79% (95% CI -1.87, -1.71; p=0)
    YLD rate - All causes        AAPC = -0.34% (95% CI -0.38, -0.30; p=0)
  [ok] tables/trend_results.csv (20 rows)


## 5. Decomposition

Das Gupta decomposition of 1990->2023 DALY change (pop size, age structure, age-specific rate) for CMNN and NCD.

In [5]:
decomp = load_script('04_decomposition')
df_decomp = decomp.run()



=== 04 DECOMPOSITION ===
  Observed vs decomposed DALY change, Vietnam:
cause_group  pop_size  age_structure  age_rate  total_decomp  observed_change direction
       CMNN   3247842       -1508711  -8184520      -6445388         -6445388  decrease
        NCD   6257108        6080731  -1710855      10626984         10626984  increase
  [ok] tables/decomposition_results.csv


## 6. SEA Comparison

NCD-share ranking (with joinpoint AAPC, delta percentage-points, and SDI-expected ratio) + YLL/YLD ratio ranking across 11 SEA countries.

In [6]:
sea = load_script('05_sea_comparison')
sea_results = sea.run()



=== 05 SEA COMPARISON ===


  SEA countries ranked by NCD share of total DALYs in 2023:
    location_name  ncd_share_1990  ncd_share_2023  delta_ncd_share_pp  aapc_ncd_share  obs_vs_expected_ratio_2023
Brunei Darussalam          76.117          81.263               5.146           0.225                       1.041
        Singapore          78.047          79.284               1.237           0.095                       0.983
         Malaysia          67.396          75.042               7.646           0.843                       0.971
          Vietnam          52.994          70.667              17.674           1.094                       1.016
         Thailand          61.325          68.376               7.052           0.455                       0.938
        Indonesia          46.917          68.037              21.120           0.774                       0.946
      Philippines          55.089          67.343              12.254           0.985                       0.939
          Myanmar          4

## 7. Figures

Seven Plotly figures saved as interactive HTML + PNG (300 dpi) + SVG.

In [7]:
figures = load_script('06_figures')
figures.run()



=== 06 FIGURES ===


  [ok] saved: figures/html/fig1_overview.html + static/fig1_overview.png/.svg


  [ok] saved: figures/html/fig2_heatmap.html + static/fig2_heatmap.png/.svg


  [ok] saved: figures/html/fig3_decomposition.html + static/fig3_decomposition.png/.svg


  [ok] saved: figures/html/fig4_sea_comparison.html + static/fig4_sea_comparison.png/.svg


  [ok] saved: figures/html/fig5_age_sex_pyramid.html + static/fig5_age_sex_pyramid.png/.svg


  [ok] saved: figures/html/fig6_yll_yld_trends.html + static/fig6_yll_yld_trends.png/.svg
  [ok] saved: figures/html/fig7_sea_yll_yld.html + static/fig7_sea_yll_yld.png/.svg


  [ok] saved: figures/html/fig8_cmnn_sensitivity.html + static/fig8_cmnn_sensitivity.png/.svg


  [ok] saved: figures/html/fig9_30q70_vietnam.html + static/fig9_30q70_vietnam.png/.svg
  [ok] saved: figures/html/fig10_sea_ncd_premature.html + static/fig10_sea_ncd_premature.png/.svg
  [ok] all 10 figures written to figures/html + figures/static


## 8. Summary

In [8]:
print('''
========================================
ANALYSIS COMPLETE
========================================
Tables  -> tables/
Figures -> figures/html/   (interactive Plotly)
           figures/static/ (PNG 300 dpi + SVG)
Processed CSVs -> data/processed/
========================================
''')



ANALYSIS COMPLETE
Tables  -> tables/
Figures -> figures/html/   (interactive Plotly)
           figures/static/ (PNG 300 dpi + SVG)
Processed CSVs -> data/processed/

